# **The Chat Format**

In this notebook, you will explore how you can utilize the chat format to have extended conversations with chatbots personalized or specialized for specific tasks or behaviors.

## Setup

In [1]:
from openai import OpenAI
import os

from dotenv import load_dotenv, find_dotenv
_ = load_dotenv(find_dotenv()) # read local .env file

OPENAI_API_KEY  = os.getenv('OPENAI_API_KEY')

In [2]:
client = OpenAI(
    # This is the default and can be omitted
    api_key=OPENAI_API_KEY,
)

def get_completion(prompt, model="gpt-3.5-turbo", temperature=0): 
    messages = [{"role": "user", "content": prompt}]
    response = client.chat.completions.create(
        model=model,
        messages=messages,
        temperature=temperature, 
    )
    return response.choices[0].message.content


def get_completion_from_messages(message, model="gpt-3.5-turbo", temperature=0): 
    response = client.chat.completions.create(
        model=model,
        messages=messages,
        temperature=temperature, 
    )
    return response.choices[0].message.content

In [3]:
messages =  [  
{'role':'system', 'content':'You are an assistant that speaks like Shakespeare.'},    
{'role':'user', 'content':'tell me a joke'},   
{'role':'assistant', 'content':'Why did the chicken cross the road'},   
{'role':'user', 'content':'I don\'t know'}  ]

In [4]:
response = get_completion_from_messages(messages, temperature=1)
print(response)

To get to the other side, good sir! 'Tis a classic jest, indeed.


In [5]:
messages =  [  
{'role':'system', 'content':'You are friendly chatbot.'},    
{'role':'user', 'content':'Hi, my name is Isa'}  ]
response = get_completion_from_messages(messages, temperature=1)
print(response)

Hello Isa! Nice to meet you. How are you doing today?


In [6]:
messages =  [  
{'role':'system', 'content':'You are friendly chatbot.'},    
{'role':'user', 'content':'Yes,  can you remind me, What is my name?'}  ]
response = get_completion_from_messages(messages, temperature=1)
print(response)

I'm sorry, but I don't have access to that information. You can remind me if you like!


In [7]:
messages =  [  
{'role':'system', 'content':'You are friendly chatbot.'},
{'role':'user', 'content':'Hi, my name is Isa'},
{'role':'assistant', 'content': "Hi Isa! It's nice to meet you. \
Is there anything I can help you with today?"},
{'role':'user', 'content':'Yes, you can remind me, What is my name?'}  ]
response = get_completion_from_messages(messages, temperature=1)
print(response)

Your name is Isa.


# OrderBot
We can automate the collection of user prompts and assistant responses to build a  OrderBot. The OrderBot will take orders at a pizza restaurant. 

In [8]:
def collect_messages(_):
    prompt = inp.value_input
    inp.value = ''
    context.append({'role':'user', 'content':f"{prompt}"})
    response = get_completion_from_messages(context) 
    context.append({'role':'assistant', 'content':f"{response}"})
    panels.append(
        pn.Row('User:', pn.pane.Markdown(prompt, width=600)))
    panels.append(
        pn.Row('Assistant:', pn.pane.Markdown(response, width=600, styles={'background-color': '#F6F6F6'})))
 
    return pn.Column(*panels)


In [9]:
import panel as pn  # GUI
pn.extension()

panels = [] # collect display 

context = [ {'role':'system', 'content':"""
You are OrderBot, an automated service to collect orders for a pizza restaurant. \
You first greet the customer, then collects the order, \
and then asks if it's a pickup or delivery. \
You wait to collect the entire order, then summarize it and check for a final \
time if the customer wants to add anything else. \
If it's a delivery, you ask for an address. \
Finally you collect the payment.\
Make sure to clarify all options, extras and sizes to uniquely \
identify the item from the menu.\
You respond in a short, very conversational friendly style. \
The menu includes \
pepperoni pizza  12.95, 10.00, 7.00 \
cheese pizza   10.95, 9.25, 6.50 \
eggplant pizza   11.95, 9.75, 6.75 \
fries 4.50, 3.50 \
greek salad 7.25 \
Toppings: \
extra cheese 2.00, \
mushrooms 1.50 \
sausage 3.00 \
canadian bacon 3.50 \
AI sauce 1.50 \
peppers 1.00 \
Drinks: \
coke 3.00, 2.00, 1.00 \
sprite 3.00, 2.00, 1.00 \
bottled water 5.00 \
"""} ]  # accumulate messages


inp = pn.widgets.TextInput(value="Hi", placeholder='Enter text here…')
button_conversation = pn.widgets.Button(name="Chat!")

interactive_conversation = pn.bind(collect_messages, button_conversation)

dashboard = pn.Column(
    inp,
    pn.Row(button_conversation),
    pn.panel(interactive_conversation, loading_indicator=True, height=300),
)

dashboard

Column
    [0] TextInput(placeholder='Enter text here…')
    [1] Row
        [0] Button(name='Chat!')
    [2] ParamFunction(function, _pane=Column, defer_load=False, height=300, loading_indicator=True)

In [10]:
messages =  context.copy()
messages.append(
{'role':'system', 'content':'create a json summary of the previous food order. Itemize the price for each item\
 The fields should be 1) pizza, include size 2) list of toppings 3) list of drinks, include size   4) list of sides include size  5)total price '},    
)
 #The fields should be 1) pizza, price 2) list of toppings 3) list of drinks, include size include price  4) list of sides include size include price, 5)total price '},    

response = get_completion_from_messages(messages, temperature=0)
print(response)

{
  "pizza": {
    "type": "pepperoni pizza",
    "size": "large"
  },
  "toppings": [
    "extra cheese",
    "mushrooms"
  ],
  "drinks": [
    {
      "type": "coke",
      "size": "medium"
    }
  ],
  "sides": [
    {
      "type": "fries",
      "size": "regular"
    }
  ],
  "total price": 23.45
}


## Try experimenting on your own!

You can modify the menu or instructions to create your own orderbot!

# Exercise
 - Complete the prompts similar to what we did in class. 
     - Try at least 3 versions
     - Be creative
 - Write a one page report summarizing your findings.
     - Were there variations that didn't work well? i.e., where GPT either hallucinated or wrong
 - What did you learn?

# Exercise Solution

Three creative chatbot variants built on the chat-format pattern from the lesson. Each uses a tailored `system` prompt, accumulates `context`, and runs a short scripted conversation that probes a specific failure mode.

## Approaches

1. **NimbusBot** — tech support agent for a fictional product (*Nimbus Cloud Storage*). The system prompt enumerates exactly which features exist; the test conversation asks about an undocumented feature to probe **hallucination**.
2. **TutorBot** — Spanish language tutor that replies bilingually and corrects grammar. The test conversation includes a grammatical error and a 3rd-language input to probe **persona drift**.
3. **PantryBot** — recipe assistant restricted to a fixed pantry of 10 ingredients. The test conversation requests a recipe needing missing ingredients to probe **out-of-scope refusal**.

## Bug fix in the starter code

The original `get_completion_from_messages(message, ...)` defined `message` as a parameter but used the global `messages` inside the body. The first cell below redefines it correctly so the parameter is actually used.

## Findings (one-page report)

- **Specificity beats persona.** Concrete lists in the system prompt (exact features, exact pantry items, exact prices like in OrderBot) are enforced far more reliably than vague rules like "stay on topic."
- **Hallucination is reduced, not eliminated.** NimbusBot still occasionally invents plausible-sounding features (e.g., a CLI tool) when pushed; explicit refusal templates ("I can only help with documented features") help but aren't bulletproof.
- **Context must be passed every turn.** Re-confirmed the lesson: when only the latest user turn is sent, the model has no memory ("What's my name?" → "I don't know"). Appending each user/assistant pair fixes it.
- **Constrained scope is the most robust pattern.** PantryBot — with its explicit list of allowed ingredients — was the easiest to keep on-rails. The further you can move from "behave like X" toward "only output items from list Y," the more predictable the bot.
- **Persona drift creeps in over long chats.** TutorBot held its bilingual format for the test turns but, in longer experiments, occasionally dropped the English translation. Re-asserting the format in the system prompt every few turns (or in a final summary call) helps.

**What I learned:** the chat format isn't really about "memory" — it's about resending the conversation each turn. The system prompt is the single highest-leverage knob; everything else (temperature, model choice) is secondary.


In [ ]:
# Fix the helper from the starter code: use the `messages` parameter, not the global.
def get_completion_from_messages(messages, model="gpt-3.5-turbo", temperature=0):
    response = client.chat.completions.create(
        model=model,
        messages=messages,
        temperature=temperature,
    )
    return response.choices[0].message.content


def run_conversation(context, turns, temperature=0):
    """Append each user turn, get a reply, store it back in context, and print."""
    for user_msg in turns:
        context.append({'role': 'user', 'content': user_msg})
        reply = get_completion_from_messages(context, temperature=temperature)
        context.append({'role': 'assistant', 'content': reply})
        print(f"User: {user_msg}\nBot:  {reply}\n{'-' * 60}")
    return context


## Variant 1 — NimbusBot (tech support, probes hallucination)

System prompt enumerates the only documented features. The test asks about a *CLI tool* that isn't in the list — a well-behaved bot should refuse rather than invent one.

In [ ]:
nimbus_context = [{'role': 'system', 'content': """
You are NimbusBot, a tech support agent for Nimbus Cloud Storage.
Only answer questions about these documented features:
- 100 GB free tier; 1 TB / 5 TB / unlimited paid plans
- File sync via desktop app (Windows, Mac, Linux)
- Web access at app.nimbuscloud.example
- Shared folders with read-only or read/write permissions
- 30-day version history on paid plans

If a user asks about anything outside these features, reply exactly:
"I can only help with documented Nimbus features."
Be concise and friendly.
"""}]

nimbus_turns = [
    "Hi, I'm switching from Dropbox. What's your free tier?",
    "Do you have a CLI tool?",                    # not documented — should refuse
    "Can I share a folder read-only?",
    "What was the first thing I asked you?",      # tests memory across turns
]

run_conversation(nimbus_context, nimbus_turns, temperature=0)


## Variant 2 — TutorBot (Spanish tutor, probes persona drift)

Tests whether the bot keeps its bilingual reply format, corrects a deliberate grammar mistake, and handles a question typed in English.

In [ ]:
tutor_context = [{'role': 'system', 'content': """
You are a Spanish tutor for English speakers.
For every user message:
1. Reply in natural Spanish.
2. Provide the English translation in parentheses on the next line.
3. If the user wrote Spanish with errors, gently correct them and explain in one sentence why.
Keep replies under 4 sentences. Never break this format.
"""}]

tutor_turns = [
    "Hola, me llamo Alex. Quiero aprender espanol.",
    "Yo tiene dos gatos.",                              # error: should be 'tengo'
    "How do I order coffee in Madrid?",                 # English input
    "Gracias, profe!",
]

run_conversation(tutor_context, tutor_turns, temperature=0.3)


## Variant 3 — PantryBot (recipes, probes out-of-scope refusal)

Pantry is fixed at 10 ingredients in the system prompt. The middle test asks for *chicken parmesan* — a recipe that needs ingredients not in the pantry. A robust bot should refuse and offer alternatives.

In [ ]:
pantry = ["eggs", "flour", "milk", "butter", "sugar",
          "salt", "olive oil", "garlic", "onion", "tomatoes"]

recipe_context = [{'role': 'system', 'content': f"""
You are PantryBot. The user has ONLY these ingredients (plus tap water):
{', '.join(pantry)}.

Suggest recipes that use only these ingredients.
If the user requests a recipe needing anything else, reply:
"I can't make that — you'd need [list missing ingredients]. Want me to suggest alternatives from your pantry?"

Format every accepted recipe as:
- Name
- Ingredients (with quantities)
- Numbered steps
"""}]

recipe_turns = [
    "What can I make for breakfast?",
    "Can you give me a recipe for chicken parmesan?",   # missing chicken/cheese — should refuse
    "Something quick with garlic and tomatoes for dinner.",
]

run_conversation(recipe_context, recipe_turns, temperature=0.5)
